In [0]:
// Databricks Scala cell: simple custom Catalyst rule demo (safe & minimal)

// imports we actually need
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.catalyst.plans.logical.{LogicalPlan, Filter}
import org.apache.spark.sql.catalyst.rules.Rule
import org.apache.spark.sql.catalyst.expressions.Literal

// Define a simple rule that removes Filter(true, child) => child
class RemoveTrueFilterRule extends Rule[LogicalPlan] {
  println(">>> RemoveTrueFilterRule created!")

    // The actual rule logic
  def apply(plan: LogicalPlan): LogicalPlan = plan transform {
    case Filter(cond, child) if cond == Literal(true) =>
      println(">>> RemoveTrueFilterRule applied!") // proof rule fires
      child
  }
  
  override def toString: String = "RemoveTrueFilterRule"
}

// Register the rule
val myRule = new RemoveTrueFilterRule()
// Note: spark.experimental.extraOptimizations expects Seq[Rule[LogicalPlan]]
spark.experimental.extraOptimizations = Seq(myRule)

// Demo DataFrame with a no-op filter
val df = spark.range(0, 100).toDF("id")
val filtered = df.filter("true")  // intentionally silly filter

println("=== explain() (compile-time plan) ===")
filtered.explain(true)

// Trigger execution so optimizer runs and rule is applied
println("\n=== Trigger execution ===")
filtered.count()

// Show executedPlan (final physical plan after optimizer / AQE)
println("\n=== executedPlan (final plan) ===")
println(filtered.queryExecution.executedPlan.toString())

// Cleanup: remove the rule if you want
// spark.experimental.extraOptimizations = Seq.empty


>>> RemoveTrueFilterRule created!
=== explain() (compile-time plan) ===
== Parsed Logical Plan ==
Filter true
+- Project [id#204L AS id#206L]
 +- Range (0, 100, step=1, splits=Some(8))

== Analyzed Logical Plan ==
id: bigint
Filter true
+- Project [id#204L AS id#206L]
 +- Range (0, 100, step=1, splits=Some(8))

== Optimized Logical Plan ==
Range (0, 100, step=1, splits=Some(8))

== Physical Plan ==
*(1) Range (0, 100, step=1, splits=8)


=== Trigger execution ===

=== executedPlan (final plan) ===
*(1) Range (0, 100, step=1, splits=8)

import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.catalyst.plans.logical.{LogicalPlan, Filter}
import org.apache.spark.sql.catalyst.rules.Rule
import org.apache.spark.sql.catalyst.expressions.Literal
defined class RemoveTrueFilterRule
myRule: RemoveTrueFilterRule = RemoveTrueFilterRule
spark.experimental.extraOptimizations: Seq[org.apache.spark.sql.catalyst.rules.Rule[org.apache.spark.sql.catalyst.plans.logical.LogicalPlan]] = List(RemoveTrueFilterRule)
df: org.apache.spark.sql.DataFrame = [id: bigint]
filtered: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [id: bigint]

In [0]:
// Databricks Scala cell: Verbose rule (fixed simpleString usage)

import org.apache.spark.sql.catalyst.plans.logical.{LogicalPlan, Filter}
import org.apache.spark.sql.catalyst.rules.Rule
import org.apache.spark.sql.catalyst.expressions.Literal

// A more verbose rule: logs when constructed, logs every Filter visited, and removes Filter(true) when matched.
class VerboseRemoveTrueFilterRule extends Rule[LogicalPlan] {

  // constructor proof
  println(">>> VerboseRemoveTrueFilterRule created!")

  def apply(plan: LogicalPlan): LogicalPlan = {
    plan transform {
      case f @ Filter(cond, child) =>
        // call simpleString() (note the parentheses) to get a concise representation of the condition
        val condDesc = try { cond.simpleString(10)  } catch { case _: Throwable => cond.toString }
        println(s">>> VerboseRemoveTrueFilterRule visited Filter with cond: $condDesc")

        // robust detection of boolean literal true
        cond match {
          case lit: Literal if lit.value == true =>
            println(">>> VerboseRemoveTrueFilterRule applied: removing Filter(true)")
            child
          case _ =>
            // not a literal true — keep it as-is
            f
        }
    }
  }

  override def toString: String = "VerboseRemoveTrueFilterRule"
}



import org.apache.spark.sql.catalyst.plans.logical.{LogicalPlan, Filter}
import org.apache.spark.sql.catalyst.rules.Rule
import org.apache.spark.sql.catalyst.expressions.Literal
defined class VerboseRemoveTrueFilterRule

In [0]:
// Register the verbose rule
val myVerboseRule = new VerboseRemoveTrueFilterRule()
spark.experimental.extraOptimizations = Seq(myVerboseRule)

// Demo DF with a no-op filter
val df = spark.range(0, 100).toDF("id")
val filtered = df.filter("true")  // silly filter

println("=== explain() (compile-time plan) ===")
filtered.explain(true)

// Trigger execution to run optimizer and apply rules
println("\n=== Trigger execution ===")
filtered.count()

println("\n=== executedPlan (final plan) ===")
println(filtered.queryExecution.executedPlan.toString())


>>> VerboseRemoveTrueFilterRule created!
=== explain() (compile-time plan) ===
== Parsed Logical Plan ==
Filter true
+- Project [id#214L AS id#216L]
 +- Range (0, 100, step=1, splits=Some(8))

== Analyzed Logical Plan ==
id: bigint
Filter true
+- Project [id#214L AS id#216L]
 +- Range (0, 100, step=1, splits=Some(8))

== Optimized Logical Plan ==
Range (0, 100, step=1, splits=Some(8))

== Physical Plan ==
*(1) Range (0, 100, step=1, splits=8)


=== Trigger execution ===

=== executedPlan (final plan) ===
*(1) Range (0, 100, step=1, splits=8)

myVerboseRule: VerboseRemoveTrueFilterRule = VerboseRemoveTrueFilterRule
spark.experimental.extraOptimizations: Seq[org.apache.spark.sql.catalyst.rules.Rule[org.apache.spark.sql.catalyst.plans.logical.LogicalPlan]] = List(VerboseRemoveTrueFilterRule)
df: org.apache.spark.sql.DataFrame = [id: bigint]
filtered: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [id: bigint]

In [0]:
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.catalyst.rules.Rule

class DebugAllPlansRule extends Rule[LogicalPlan] {
  println(">>> DebugAllPlansRule created!")   // constructor proof

  def apply(plan: LogicalPlan): LogicalPlan = {
    println(s">>> DebugAllPlansRule visiting plan node: ${plan.nodeName}")
    plan   // no transform, just observe
  }

  override def toString: String = "DebugAllPlansRule"
}

// register
val debugRule = new DebugAllPlansRule()
spark.experimental.extraOptimizations = Seq(debugRule)

// trigger
spark.range(0,10).count()


>>> DebugAllPlansRule created!
>>> DebugAllPlansRule visiting plan node: Aggregate
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.catalyst.rules.Rule
defined class DebugAllPlansRule
debugRule: DebugAllPlansRule = DebugAllPlansRule
spark.experimental.extraOptimizations: Seq[org.apache.spark.sql.catalyst.rules.Rule[org.apache.spark.sql.catalyst.plans.logical.LogicalPlan]] = List(DebugAllPlansRule)
res4: Long = 10

In [0]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.SparkSessionExtensions
import org.apache.spark.sql.catalyst.rules.Rule
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.catalyst.plans.logical.Filter
import org.apache.spark.sql.catalyst.expressions.Literal

// The rule that removes Filter(true)
class RemoveTrueFilterRule extends Rule[LogicalPlan] {
  println(">>> RemoveTrueFilterRule created (via extensions)!")
  def apply(plan: LogicalPlan): LogicalPlan = plan transform {
    case f @ Filter(cond, child) =>
      val desc = try { cond.simpleString(10) } catch { case _: Throwable => cond.toString }
      println(s">>> RemoveTrueFilterRule visited Filter with cond: $desc")
      cond match {
        case lit: Literal if lit.value == true =>
          println(">>> RemoveTrueFilterRule applied: removing Filter(true)")
          child
        case _ => f
      }
  }
  override def toString: String = "RemoveTrueFilterRule"
}

// extension function to inject optimizer rule
val extensions = (e: SparkSessionExtensions) => {
  e.injectOptimizerRule { session => new RemoveTrueFilterRule().asInstanceOf[Rule[LogicalPlan]] }
}

// Create a NEW SparkSession with the extension
val spark2 = SparkSession.builder()
  .appName("WithCustomRule")
  .master("local[*]")        // or your cluster master
  .withExtensions(extensions)
  .getOrCreate()

// Use spark2 to run your demo so the rule is injected early
import spark2.implicits._
val df2 = spark2.range(0,100).toDF("id")
val filtered2 = df2.filter("true")
filtered2.explain(true)
filtered2.count()


>>> DebugAllPlansRule visiting plan node: Range
== Parsed Logical Plan ==
Filter true
+- Project [id#232L AS id#234L]
 +- Range (0, 100, step=1, splits=Some(8))

== Analyzed Logical Plan ==
id: bigint
Filter true
+- Project [id#232L AS id#234L]
 +- Range (0, 100, step=1, splits=Some(8))

== Optimized Logical Plan ==
Range (0, 100, step=1, splits=Some(8))

== Physical Plan ==
*(1) Range (0, 100, step=1, splits=8)

>>> DebugAllPlansRule visiting plan node: Aggregate
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.SparkSessionExtensions
import org.apache.spark.sql.catalyst.rules.Rule
import org.apache.spark.sql.catalyst.plans.logical.LogicalPlan
import org.apache.spark.sql.catalyst.plans.logical.Filter
import org.apache.spark.sql.catalyst.expressions.Literal
defined class RemoveTrueFilterRule
extensions: org.apache.spark.sql.SparkSessionExtensions => Unit = $Lambda$8540/1939923135@13ff396d
spark2: org.apache.spark.sql.SparkSession = org.apache.spark.sql.SparkSession@c2c9799
import spark2.implicits._
df2: org.apache.spark.sql.DataFrame = [id: bigint]
filtered2: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [id: bigint]
res5: Long = 100